# Notebook 11 – EDA Visualization
## Exploratory Data Analysis (EDA) using Matplotlib and Seaborn

**Objective:** Create meaningful visualizations, explain why each chart is selected, interpret its pattern, and record the insight obtained.


### Chart selection guide
| Question | Suitable visualization |
|---|---|
| How is one numerical variable distributed? | Histogram, distribution plot |
| Are there possible outliers? | Box plot |
| Which categories are most frequent? | Bar chart, count plot |
| Do two numerical variables relate? | Scatter plot |
| How does a metric change over time? | Line plot |
| Do distributions differ across groups? | Violin plot |
| How do several numerical features relate? | Pair plot, correlation heatmap |


## 1. Import libraries

Install the libraries if needed by running `pip install pandas numpy matplotlib seaborn` in a notebook cell or terminal.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 13

print("Libraries imported successfully.")

## 2. Load the dataset

The example below creates a reproducible sample sales dataset. To use your own dataset, uncomment the `pd.read_csv(...)` line and provide the correct file path.

In [ ]:
# Example dataset: replace this section with your own dataset when ready.
rng = np.random.default_rng(42)
n = 250

dates = pd.date_range("2025-01-01", periods=n, freq="D")
categories = rng.choice(["Electronics", "Clothing", "Home", "Books"], size=n,
                        p=[0.30, 0.28, 0.24, 0.18])
regions = rng.choice(["North", "South", "East", "West"], size=n)
units = rng.integers(1, 10, size=n)
base_price = pd.Series(categories).map({
    "Electronics": 180, "Clothing": 45, "Home": 75, "Books": 20
}).to_numpy()
price = np.maximum(5, base_price + rng.normal(0, base_price * 0.18, n))
discount = rng.uniform(0, 0.25, n)
revenue = units * price * (1 - discount)

df = pd.DataFrame({
    "Date": dates,
    "Category": categories,
    "Region": regions,
    "Units": units,
    "UnitPrice": price.round(2),
    "Discount": discount.round(3),
    "Revenue": revenue.round(2)
})

# For your own CSV dataset, use for example:
# df = pd.read_csv("your_dataset.csv")

df.head()

## 3. Understand the available columns

Check the dataset shape, column types, and basic statistics before choosing charts. This helps prevent unsuitable visualizations.

In [ ]:
print("Dataset shape (rows, columns):", df.shape)
display(df.info())
display(df.describe(include="all").T)

In [ ]:
# Identify numerical and categorical columns
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
datetime_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()

print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Datetime columns:", datetime_cols)

## 4. Configure columns for visualization

Set the columns below to match your dataset. If a selected column is unavailable, the relevant chart will be skipped rather than producing a misleading graph.

In [ ]:
# Update these names to match your dataset.
NUMERIC_COLUMN = "Revenue"       # e.g., "Age", "Salary", "Revenue"
SECOND_NUMERIC_COLUMN = "Units"  # e.g., "Income", "Spending"
CATEGORY_COLUMN = "Category"     # e.g., "Department", "Product"
GROUP_COLUMN = "Region"          # optional grouping variable
DATE_COLUMN = "Date"             # optional date/time column

def has_column(column):
    return column in df.columns

def explain_chart(title, why, interpretation, insight):
    print(f"WHY THIS CHART: {why}")
    print(f"INTERPRETATION: {interpretation}")
    print(f"INSIGHT TO RECORD: {insight}")


## 5. Histogram

**Why selected:** A histogram groups a numerical variable into intervals and shows its distribution, common ranges, skewness, and possible unusual values.

**Interpretation:** Look for the most common range, symmetry or skewness, multiple peaks, and gaps. The exact interpretation depends on the plotted dataset.

In [ ]:
if has_column(NUMERIC_COLUMN) and pd.api.types.is_numeric_dtype(df[NUMERIC_COLUMN]):
    values = df[NUMERIC_COLUMN].dropna()
    if not values.empty:
        fig, ax = plt.subplots()
        sns.histplot(data=df, x=NUMERIC_COLUMN, bins="auto", kde=False, ax=ax)
        ax.set_title(f"Histogram of {NUMERIC_COLUMN}")
        ax.set_xlabel(NUMERIC_COLUMN)
        ax.set_ylabel("Number of observations")
        plt.tight_layout()
        plt.show()
        explain_chart(
            "Histogram",
            f"To see how {NUMERIC_COLUMN} values are distributed across ranges.",
            "Identify the tallest bars, whether the shape is symmetric or skewed, and whether there are gaps or multiple peaks.",
            f"Record the most common range and the overall distribution shape for {NUMERIC_COLUMN}."
        )
    else:
        print("Skipped: no non-missing values.")
else:
    print(f"Skipped: {NUMERIC_COLUMN!r} is not a numeric column in this dataset.")

## 6. Box plot

**Why selected:** A box plot summarizes the median, quartiles, spread, and potential outliers for a numerical variable.

**Interpretation:** Points beyond the whiskers may be potential outliers. They are not automatically errors; verify them against the source and business context.

In [ ]:
if has_column(NUMERIC_COLUMN) and pd.api.types.is_numeric_dtype(df[NUMERIC_COLUMN]):
    fig, ax = plt.subplots()
    sns.boxplot(data=df, x=NUMERIC_COLUMN, ax=ax)
    ax.set_title(f"Box Plot of {NUMERIC_COLUMN}")
    plt.tight_layout()
    plt.show()
    q1, q3 = df[NUMERIC_COLUMN].quantile([0.25, 0.75])
    iqr = q3 - q1
    potential = df[(df[NUMERIC_COLUMN] < q1 - 1.5 * iqr) |
                   (df[NUMERIC_COLUMN] > q3 + 1.5 * iqr)][NUMERIC_COLUMN]
    explain_chart(
        "Box plot",
        f"To summarize the spread of {NUMERIC_COLUMN} and flag possible outliers.",
        f"The line inside the box is the median; the box spans the middle 50% of values. Potential IQR outliers detected: {len(potential)}.",
        f"Review the {len(potential)} potential outlier(s). Confirm whether they are valid observations before deciding to keep or correct them."
    )
else:
    print(f"Skipped: {NUMERIC_COLUMN!r} is not a numeric column.")

## 7. Bar chart

**Why selected:** A bar chart compares a numeric summary across categories, such as total revenue by product category.

**Interpretation:** Compare bar heights and identify the largest and smallest category values. Use aggregation that answers the business question.

In [ ]:
if has_column(CATEGORY_COLUMN) and has_column(NUMERIC_COLUMN):
    if pd.api.types.is_numeric_dtype(df[NUMERIC_COLUMN]):
        summary = df.groupby(CATEGORY_COLUMN, dropna=False)[NUMERIC_COLUMN].mean().sort_values(ascending=False)
        if not summary.empty:
            fig, ax = plt.subplots()
            summary.plot(kind="bar", ax=ax)
            ax.set_title(f"Average {NUMERIC_COLUMN} by {CATEGORY_COLUMN}")
            ax.set_xlabel(CATEGORY_COLUMN)
            ax.set_ylabel(f"Average {NUMERIC_COLUMN}")
            plt.xticks(rotation=35, ha="right")
            plt.tight_layout()
            plt.show()
            explain_chart(
                "Bar chart",
                f"To compare average {NUMERIC_COLUMN} across {CATEGORY_COLUMN} groups.",
                f"Categories with taller bars have higher average {NUMERIC_COLUMN} values in this dataset.",
                f"Highest average: {summary.index[0]} ({summary.iloc[0]:.2f}); lowest average: {summary.index[-1]} ({summary.iloc[-1]:.2f})."
            )
    else:
        print("Skipped: the selected measure must be numeric.")
else:
    print("Skipped: configure valid category and numeric columns.")

## 8. Count plot

**Why selected:** A count plot shows how many records belong to each category. It is useful for checking class/category imbalance.

**Interpretation:** Large differences in bar heights indicate that some categories are much more common than others.

In [ ]:
if has_column(CATEGORY_COLUMN):
    counts = df[CATEGORY_COLUMN].value_counts(dropna=False)
    fig, ax = plt.subplots()
    sns.countplot(data=df, x=CATEGORY_COLUMN, order=df[CATEGORY_COLUMN].value_counts().index, ax=ax)
    ax.set_title(f"Count of Records by {CATEGORY_COLUMN}")
    ax.set_xlabel(CATEGORY_COLUMN)
    ax.set_ylabel("Record count")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()
    explain_chart(
        "Count plot",
        f"To understand the frequency and balance of {CATEGORY_COLUMN} values.",
        f"The tallest bar is the most frequent category; the shortest bar is the least frequent category.",
        f"Most frequent category: {counts.index[0]} ({counts.iloc[0]} records). Check whether rare categories need special handling."
    )
else:
    print("Skipped: configure a valid categorical column.")

## 9. Scatter plot

**Why selected:** A scatter plot helps assess the relationship between two numerical variables and can reveal trends, clusters, and unusual observations.

**Interpretation:** An upward pattern suggests a positive association; a downward pattern suggests a negative association. A pattern alone does not prove causation.

In [ ]:
if (has_column(NUMERIC_COLUMN) and has_column(SECOND_NUMERIC_COLUMN)
    and pd.api.types.is_numeric_dtype(df[NUMERIC_COLUMN])
    and pd.api.types.is_numeric_dtype(df[SECOND_NUMERIC_COLUMN])):
    plot_df = df[[SECOND_NUMERIC_COLUMN, NUMERIC_COLUMN]].dropna()
    fig, ax = plt.subplots()
    sns.scatterplot(data=plot_df, x=SECOND_NUMERIC_COLUMN, y=NUMERIC_COLUMN, alpha=0.7, ax=ax)
    ax.set_title(f"{NUMERIC_COLUMN} vs {SECOND_NUMERIC_COLUMN}")
    plt.tight_layout()
    plt.show()
    corr = plot_df[SECOND_NUMERIC_COLUMN].corr(plot_df[NUMERIC_COLUMN])
    explain_chart(
        "Scatter plot",
        f"To check whether {SECOND_NUMERIC_COLUMN} and {NUMERIC_COLUMN} move together.",
        f"The Pearson correlation for the plotted rows is {corr:.3f}. Inspect the point pattern as well as this summary.",
        "Describe whether the relationship looks positive, negative, weak, strong, or nonlinear. Correlation does not establish causation."
    )
else:
    print("Skipped: configure two valid numeric columns.")

## 10. Line plot

**Why selected:** A line plot is suitable when observations have a meaningful order, especially dates or time periods. It can reveal trends and seasonal changes.

**Interpretation:** First aggregate to the time frequency relevant to the question; connecting unsorted or unordered observations can be misleading.

In [ ]:
if has_column(DATE_COLUMN) and has_column(NUMERIC_COLUMN):
    line_df = df[[DATE_COLUMN, NUMERIC_COLUMN]].copy()
    line_df[DATE_COLUMN] = pd.to_datetime(line_df[DATE_COLUMN], errors="coerce")
    line_df = line_df.dropna().sort_values(DATE_COLUMN)
    if not line_df.empty and pd.api.types.is_numeric_dtype(line_df[NUMERIC_COLUMN]):
        # Aggregate by day in case the dataset has multiple records per date.
        daily = line_df.groupby(DATE_COLUMN, as_index=False)[NUMERIC_COLUMN].mean()
        fig, ax = plt.subplots()
        sns.lineplot(data=daily, x=DATE_COLUMN, y=NUMERIC_COLUMN, marker="o", ax=ax)
        ax.set_title(f"{NUMERIC_COLUMN} over time")
        ax.set_xlabel("Date")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()
        explain_chart(
            "Line plot",
            f"To examine changes in {NUMERIC_COLUMN} across the ordered dates.",
            "Look for an overall increase/decrease, repeated cycles, sudden changes, or periods of stability.",
            "Record the visible time trend and investigate unusual spikes or drops using the underlying records."
        )
    else:
        print("Skipped: no valid date/numeric rows.")
else:
    print("Skipped: configure a valid date column and numeric measure.")

## 11. Violin plot

**Why selected:** A violin plot compares the distribution and density of a numerical variable across categories.

**Interpretation:** Wider sections represent values where observations are more concentrated. Compare medians and spread, but remember that density shape depends on the smoothing method.

In [ ]:
if (has_column(CATEGORY_COLUMN) and has_column(NUMERIC_COLUMN)
    and pd.api.types.is_numeric_dtype(df[NUMERIC_COLUMN])):
    violin_df = df[[CATEGORY_COLUMN, NUMERIC_COLUMN]].dropna()
    if not violin_df.empty:
        fig, ax = plt.subplots()
        sns.violinplot(data=violin_df, x=CATEGORY_COLUMN, y=NUMERIC_COLUMN, inner="quartile", cut=0, ax=ax)
        ax.set_title(f"Distribution of {NUMERIC_COLUMN} by {CATEGORY_COLUMN}")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()
        medians = violin_df.groupby(CATEGORY_COLUMN)[NUMERIC_COLUMN].median().sort_values(ascending=False)
        explain_chart(
            "Violin plot",
            f"To compare the shape and spread of {NUMERIC_COLUMN} across {CATEGORY_COLUMN} groups.",
            "Compare each group's median, spread, and density shape; groups may differ even if their averages are similar.",
            f"Highest group median: {medians.index[0]} ({medians.iloc[0]:.2f}). Inspect the plot for differences in spread or multiple clusters."
        )
else:
    print("Skipped: configure a category and numeric column.")

## 12. Pair plot

**Why selected:** A pair plot gives a compact overview of pairwise relationships between a small set of numerical variables and their individual distributions.

**Interpretation:** Look for pairwise trends, clusters, and possible correlations. Use only a few relevant features because pair plots become crowded with many columns.

In [ ]:
pair_cols = [c for c in numeric_cols if df[c].nunique(dropna=True) > 1][:4]
if len(pair_cols) >= 2:
    pair_df = df[pair_cols].dropna()
    if len(pair_df) > 500:
        pair_df = pair_df.sample(500, random_state=42)
    g = sns.pairplot(pair_df, diag_kind="hist", corner=True)
    g.fig.suptitle("Pair Plot of Selected Numerical Features", y=1.02)
    plt.show()
    corr_matrix = pair_df.corr(numeric_only=True)
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    strongest = upper.stack().abs().sort_values(ascending=False)
    strongest_text = (f"{strongest.index[0][0]} and {strongest.index[0][1]} "
                      f"(absolute correlation {strongest.iloc[0]:.3f})") if not strongest.empty else "no pair available"
    explain_chart(
        "Pair plot",
        "To inspect multiple numerical feature relationships without making many separate charts.",
        "Each off-diagonal panel compares two features; diagonal panels show individual distributions.",
        f"Among selected features, the strongest absolute linear correlation is for {strongest_text}. Check the panels for nonlinear patterns and clusters too."
    )
else:
    print("Skipped: at least two varying numerical columns are needed.")

## 13. Correlation heatmap

**Why selected:** A heatmap summarizes pairwise linear correlations between numerical variables and makes strongly related features easier to spot.

**Interpretation:** Correlation values range from -1 to +1. Values near +1 indicate positive linear association; values near -1 indicate negative linear association; values near 0 indicate little linear association.

In [ ]:
heatmap_cols = [c for c in numeric_cols if df[c].nunique(dropna=True) > 1]
if len(heatmap_cols) >= 2:
    corr_matrix = df[heatmap_cols].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(max(7, len(heatmap_cols) * 0.8), max(5, len(heatmap_cols) * 0.65)))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                vmin=-1, vmax=1, square=True, ax=ax)
    ax.set_title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    pairs = upper.stack()
    if not pairs.empty:
        strongest_pair = pairs.abs().idxmax()
        value = pairs.loc[strongest_pair]
        explain_chart(
            "Correlation heatmap",
            "To summarize linear relationships across numerical features.",
            f"Strongest absolute off-diagonal correlation: {strongest_pair[0]} vs {strongest_pair[1]} = {value:.3f}.",
            "Investigate strongly correlated features for redundancy or business meaning. Correlation alone does not imply causation."
        )
else:
    print("Skipped: at least two varying numeric columns are needed.")

## 14. Distribution plot (KDE)

**Why selected:** A kernel density estimate (KDE) provides a smoothed view of a numerical distribution. It can complement a histogram when comparing overall shape.

**Interpretation:** Peaks indicate regions of higher density. KDE is a smoothed estimate, so it may look different from the exact observed values, especially for small samples.

In [ ]:
if has_column(NUMERIC_COLUMN) and pd.api.types.is_numeric_dtype(df[NUMERIC_COLUMN]):
    values = df[NUMERIC_COLUMN].dropna()
    if len(values) >= 2 and values.nunique() > 1:
        fig, ax = plt.subplots()
        sns.histplot(values, stat="density", bins="auto", alpha=0.3, label="Histogram", ax=ax)
        sns.kdeplot(values, label="KDE", ax=ax)
        ax.set_title(f"Distribution of {NUMERIC_COLUMN}: Histogram + KDE")
        ax.set_xlabel(NUMERIC_COLUMN)
        ax.legend()
        plt.tight_layout()
        plt.show()
        skew = values.skew()
        explain_chart(
            "Distribution plot",
            f"To examine the shape and smooth density of {NUMERIC_COLUMN}.",
            f"Sample skewness is {skew:.3f}; positive values suggest right-skew and negative values suggest left-skew, though the plot should also be inspected.",
            "Record the dominant density region and whether the distribution appears skewed, symmetric, or multimodal."
        )
    else:
        print("Skipped: at least two distinct non-missing values are needed.")
else:
    print(f"Skipped: {NUMERIC_COLUMN!r} is not a numeric column.")

## 15. Final visualization summary

Use this checklist to summarize only the insights supported by your actual charts.

In [ ]:
summary = {
    "Dataset rows": len(df),
    "Dataset columns": df.shape[1],
    "Numerical columns available": len(numeric_cols),
    "Categorical columns available": len(categorical_cols),
    "Datetime columns available": len(datetime_cols),
    "Selected numeric column": NUMERIC_COLUMN if has_column(NUMERIC_COLUMN) else "Not found",
    "Selected category column": CATEGORY_COLUMN if has_column(CATEGORY_COLUMN) else "Not found",
}
pd.Series(summary, name="Value").to_frame()